# FoodBridge Multilingual Customer-Support Intent Classifier

**Languages:** English, Hindi, Kannada  
**Transformer:** `google-bert/bert-base-multilingual-cased` (mBERT)  
**Training data:** Custom FoodBridge examples + selected examples from Amazon Science's MASSIVE dataset.

This notebook fine-tunes a pretrained transformer for **intent classification**. It does not generate answers by itself and does not include voice input/output. Voice can be added later by converting speech to text before prediction and reading responses aloud with text-to-speech.

**Run in Google Colab with a GPU:** Runtime → Change runtime type → T4 GPU (or another available GPU), then run cells from top to bottom.


In [ ]:
!pip -q install "datasets<4" transformers accelerate scikit-learn pandas matplotlib seaborn


In [ ]:
import os
import json
import random
import shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support, f1_score
)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, select Runtime > Change runtime type > GPU.")

## 1. Build the combined training dataset

We combine two sources:

1. **Custom FoodBridge examples** for the platform's real support intents (donations, NGO claims, volunteer pickups, delivery status, etc.).
2. **MASSIVE** (Amazon Science), a public multilingual intent dataset. We use selected English (`en-US`), Hindi (`hi-IN`), and Kannada (`kn-IN`) examples to strengthen general greetings, thanks, goodbyes, and out-of-scope detection.

MASSIVE does **not** contain FoodBridge-specific intents. Its general examples are mapped only to the matching conversational labels; unrelated MASSIVE intents are used as out-of-scope examples. We cap the number imported per class/language so they do not overwhelm the smaller FoodBridge-specific classes.

Dataset source: [AmazonScience/MASSIVE on Hugging Face](https://huggingface.co/datasets/AmazonScience/massive). The dataset is licensed CC BY 4.0; cite the dataset and its paper in your assignment report.


In [ ]:
examples = {
    "greeting": {
        "en": ["Hello", "Hi, can you help me?", "Good morning", "Hey FoodBridge"],
        "hi": ["नमस्ते", "क्या आप मेरी मदद कर सकते हैं?", "सुप्रभात", "हैलो फूडब्रिज"],
        "kn": ["ನಮಸ್ಕಾರ", "ನೀವು ನನಗೆ ಸಹಾಯ ಮಾಡಬಹುದೇ?", "ಶುಭೋದಯ", "ಹಲೋ ಫುಡ್‌ಬ್ರಿಡ್ಜ್"]
    },
    "goodbye": {
        "en": ["Goodbye", "Bye for now", "See you later", "Have a nice day"],
        "hi": ["अलविदा", "अभी के लिए चलता हूँ", "फिर मिलेंगे", "आपका दिन शुभ हो"],
        "kn": ["ವಿದಾಯ", "ಈಗ ಹೋಗುತ್ತೇನೆ", "ಮತ್ತೆ ಭೇಟಿಯಾಗೋಣ", "ನಿಮ್ಮ ದಿನ ಚೆನ್ನಾಗಿರಲಿ"]
    },
    "thanks": {
        "en": ["Thank you", "Thanks for your help", "That was helpful", "I appreciate your assistance"],
        "hi": ["धन्यवाद", "मदद के लिए शुक्रिया", "यह उपयोगी था", "आपकी सहायता के लिए आभारी हूँ"],
        "kn": ["ಧನ್ಯವಾದಗಳು", "ಸಹಾಯಕ್ಕಾಗಿ ಧನ್ಯವಾದಗಳು", "ಇದು ಉಪಯುಕ್ತವಾಗಿತ್ತು", "ನಿಮ್ಮ ಸಹಾಯಕ್ಕೆ ಕೃತಜ್ಞತೆಗಳು"]
    },
    "about_foodbridge": {
        "en": ["What is FoodBridge?", "Tell me about this platform", "How does FoodBridge work?", "What is the purpose of FoodBridge?"],
        "hi": ["फूडब्रिज क्या है?", "यह प्लेटफॉर्म क्या करता है?", "फूडब्रिज कैसे काम करता है?", "फूडब्रिज का उद्देश्य क्या है?"],
        "kn": ["ಫುಡ್‌ಬ್ರಿಡ್ಜ್ ಎಂದರೇನು?", "ಈ ವೇದಿಕೆ ಏನು ಮಾಡುತ್ತದೆ?", "ಫುಡ್‌ಬ್ರಿಡ್ಜ್ ಹೇಗೆ ಕೆಲಸ ಮಾಡುತ್ತದೆ?", "ಫುಡ್‌ಬ್ರಿಡ್ಜ್‌ನ ಉದ್ದೇಶವೇನು?"]
    },
    "donor_registration": {
        "en": ["How can I register as a food donor?", "I want to register my restaurant", "How do I become a donor?", "Help me create a donor account"],
        "hi": ["मैं भोजन दाता के रूप में पंजीकरण कैसे करूँ?", "मैं अपना रेस्टोरेंट रजिस्टर करना चाहता हूँ", "मैं दाता कैसे बन सकता हूँ?", "मुझे डोनर अकाउंट बनाना है"],
        "kn": ["ಆಹಾರ ದಾನಿಯಾಗಿ ನೋಂದಾಯಿಸಿಕೊಳ್ಳುವುದು ಹೇಗೆ?", "ನನ್ನ ರೆಸ್ಟೋರೆಂಟ್ ನೋಂದಾಯಿಸಬೇಕು", "ನಾನು ದಾನಿಯಾಗುವುದು ಹೇಗೆ?", "ನನಗೆ ದಾನಿ ಖಾತೆ ಬೇಕು"]
    },
    "create_donation": {
        "en": ["How do I post surplus food?", "I want to create a food donation", "How can my restaurant donate extra food?", "Where can I list available food?"],
        "hi": ["बचा हुआ खाना कैसे पोस्ट करूँ?", "मुझे भोजन दान की जानकारी डालनी है", "मेरा रेस्टोरेंट अतिरिक्त खाना दान करना चाहता है", "उपलब्ध भोजन कहाँ सूचीबद्ध करूँ?"],
        "kn": ["ಉಳಿದ ಆಹಾರವನ್ನು ಹೇಗೆ ಪೋಸ್ಟ್ ಮಾಡುವುದು?", "ನಾನು ಆಹಾರ ದಾನವನ್ನು ಸೇರಿಸಬೇಕು", "ನನ್ನ ರೆಸ್ಟೋರೆಂಟ್ ಹೆಚ್ಚುವರಿ ಆಹಾರ ದಾನ ಮಾಡಲು ಬಯಸುತ್ತದೆ", "ಲಭ್ಯವಿರುವ ಆಹಾರವನ್ನು ಎಲ್ಲಿ ಪಟ್ಟಿ ಮಾಡಬಹುದು?"]
    },
    "donation_expiry": {
        "en": ["How do I set the food expiry time?", "When does my donation expire?", "Why is the expiry time important?", "How can I check donation expiry?"],
        "hi": ["भोजन की समाप्ति का समय कैसे सेट करें?", "मेरा दान कब समाप्त होगा?", "समाप्ति का समय क्यों जरूरी है?", "दान की समाप्ति कैसे जाँचें?"],
        "kn": ["ಆಹಾರದ ಮುಕ್ತಾಯ ಸಮಯವನ್ನು ಹೇಗೆ ಹೊಂದಿಸುವುದು?", "ನನ್ನ ದಾನದ ಅವಧಿ ಯಾವಾಗ ಮುಗಿಯುತ್ತದೆ?", "ಮುಕ್ತಾಯ ಸಮಯ ಏಕೆ ಮುಖ್ಯ?", "ದಾನದ ಮುಕ್ತಾಯವನ್ನು ಹೇಗೆ ಪರಿಶೀಲಿಸುವುದು?"]
    },
    "claim_donation": {
        "en": ["How can an NGO claim a donation?", "I want to accept available food", "How do we claim surplus food?", "Where can I find food donations to claim?"],
        "hi": ["एनजीओ दान का दावा कैसे कर सकता है?", "मैं उपलब्ध भोजन स्वीकार करना चाहता हूँ", "हम अतिरिक्त भोजन कैसे प्राप्त करें?", "भोजन दान कहाँ मिलेंगे?"],
        "kn": ["ಎನ್‌ಜಿಒ ದಾನವನ್ನು ಹೇಗೆ ಸ್ವೀಕರಿಸಬಹುದು?", "ನಾನು ಲಭ್ಯವಿರುವ ಆಹಾರವನ್ನು ಸ್ವೀಕರಿಸಲು ಬಯಸುತ್ತೇನೆ", "ಹೆಚ್ಚುವರಿ ಆಹಾರವನ್ನು ಹೇಗೆ ಪಡೆಯುವುದು?", "ಆಹಾರ ದಾನಗಳು ಎಲ್ಲಿ ಲಭ್ಯವಿವೆ?"]
    },
    "ngo_approval": {
        "en": ["How can I check my NGO approval?", "Why is my NGO registration pending?", "How does NGO verification work?", "When will my NGO account be approved?"],
        "hi": ["मैं अपने एनजीओ की मंजूरी कैसे जाँचूँ?", "मेरा एनजीओ पंजीकरण लंबित क्यों है?", "एनजीओ का सत्यापन कैसे होता है?", "मेरा एनजीओ अकाउंट कब मंजूर होगा?"],
        "kn": ["ನನ್ನ ಎನ್‌ಜಿಒ ಅನುಮೋದನೆಯನ್ನು ಹೇಗೆ ಪರಿಶೀಲಿಸಬೇಕು?", "ನನ್ನ ಎನ್‌ಜಿಒ ನೋಂದಣಿ ಏಕೆ ಬಾಕಿಯಿದೆ?", "ಎನ್‌ಜಿಒ ಪರಿಶೀಲನೆ ಹೇಗೆ ನಡೆಯುತ್ತದೆ?", "ನನ್ನ ಎನ್‌ಜಿಒ ಖಾತೆ ಯಾವಾಗ ಅನುಮೋದನೆಯಾಗುತ್ತದೆ?"]
    },
    "volunteer_registration": {
        "en": ["How do I register as a volunteer driver?", "I want to volunteer for pickups", "How can I join as a delivery volunteer?", "Where can I create a volunteer account?"],
        "hi": ["स्वयंसेवी ड्राइवर के रूप में कैसे पंजीकरण करूँ?", "मैं भोजन लेने के लिए स्वयंसेवा करना चाहता हूँ", "डिलीवरी स्वयंसेवक कैसे बनें?", "स्वयंसेवक अकाउंट कहाँ बनाएँ?"],
        "kn": ["ಸ್ವಯಂಸೇವಕ ಚಾಲಕನಾಗಿ ನೋಂದಾಯಿಸುವುದು ಹೇಗೆ?", "ಆಹಾರ ಸಂಗ್ರಹಣೆಗೆ ಸ್ವಯಂಸೇವಕನಾಗಲು ಬಯಸುತ್ತೇನೆ", "ವಿತರಣಾ ಸ್ವಯಂಸೇವಕನಾಗುವುದು ಹೇಗೆ?", "ಸ್ವಯಂಸೇವಕ ಖಾತೆಯನ್ನು ಎಲ್ಲಿ ರಚಿಸಬಹುದು?"]
    },
    "pickup_assignment": {
        "en": ["How do I check my pickup assignment?", "Where can I see my assigned pickup?", "How are volunteer drivers assigned?", "I need information about my pickup task"],
        "hi": ["मैं अपना पिकअप असाइनमेंट कैसे देखूँ?", "मेरा निर्धारित पिकअप कहाँ दिखेगा?", "स्वयंसेवी ड्राइवर कैसे नियुक्त होते हैं?", "मुझे अपने पिकअप कार्य की जानकारी चाहिए"],
        "kn": ["ನನ್ನ ಪಿಕಪ್ ನಿಯೋಜನೆಯನ್ನು ಹೇಗೆ ನೋಡುವುದು?", "ನನಗೆ ನೀಡಲಾದ ಪಿಕಪ್ ಎಲ್ಲಿ ಕಾಣುತ್ತದೆ?", "ಸ್ವಯಂಸೇವಕ ಚಾಲಕರನ್ನು ಹೇಗೆ ನಿಯೋಜಿಸುತ್ತಾರೆ?", "ನನ್ನ ಪಿಕಪ್ ಕೆಲಸದ ಮಾಹಿತಿ ಬೇಕು"]
    },
    "delivery_status": {
        "en": ["How can I check delivery status?", "Has the food been delivered?", "Where can I see the delivery progress?", "How do I update a delivery status?"],
        "hi": ["डिलीवरी की स्थिति कैसे जाँचूँ?", "क्या भोजन पहुँचा दिया गया है?", "डिलीवरी की प्रगति कहाँ देखूँ?", "डिलीवरी की स्थिति कैसे अपडेट करूँ?"],
        "kn": ["ವಿತರಣೆಯ ಸ್ಥಿತಿಯನ್ನು ಹೇಗೆ ಪರಿಶೀಲಿಸುವುದು?", "ಆಹಾರ ತಲುಪಿಸಲಾಗಿದೆಯೇ?", "ವಿತರಣೆಯ ಪ್ರಗತಿಯನ್ನು ಎಲ್ಲಿ ನೋಡಬಹುದು?", "ವಿತರಣೆಯ ಸ್ಥಿತಿಯನ್ನು ಹೇಗೆ ನವೀಕರಿಸಬೇಕು?"]
    },
    "donation_history": {
        "en": ["Where can I see my donation history?", "Show me my previous donations", "How can I check past donation records?", "I want to view my donation activity"],
        "hi": ["मैं अपने दान का इतिहास कहाँ देखूँ?", "मेरे पिछले दान दिखाएँ", "पुराने दान के रिकॉर्ड कैसे जाँचूँ?", "मैं अपनी दान गतिविधि देखना चाहता हूँ"],
        "kn": ["ನನ್ನ ದಾನದ ಇತಿಹಾಸವನ್ನು ಎಲ್ಲಿ ನೋಡಬಹುದು?", "ನನ್ನ ಹಿಂದಿನ ದಾನಗಳನ್ನು ತೋರಿಸಿ", "ಹಿಂದಿನ ದಾನ ದಾಖಲೆಗಳನ್ನು ಹೇಗೆ ಪರಿಶೀಲಿಸುವುದು?", "ನನ್ನ ದಾನ ಚಟುವಟಿಕೆಯನ್ನು ನೋಡಬೇಕು"]
    },
    "account_login": {
        "en": ["I cannot log into my account", "I forgot my password", "Why is my login not working?", "I am having trouble signing in"],
        "hi": ["मैं अपने अकाउंट में लॉगिन नहीं कर पा रहा हूँ", "मैं अपना पासवर्ड भूल गया हूँ", "मेरा लॉगिन काम क्यों नहीं कर रहा?", "मुझे साइन इन करने में समस्या है"],
        "kn": ["ನನ್ನ ಖಾತೆಗೆ ಲಾಗಿನ್ ಆಗಲು ಸಾಧ್ಯವಾಗುತ್ತಿಲ್ಲ", "ನನ್ನ ಪಾಸ್‌ವರ್ಡ್ ಮರೆತಿದ್ದೇನೆ", "ನನ್ನ ಲಾಗಿನ್ ಏಕೆ ಕೆಲಸ ಮಾಡುತ್ತಿಲ್ಲ?", "ಸೈನ್ ಇನ್ ಮಾಡಲು ಸಮಸ್ಯೆಯಾಗುತ್ತಿದೆ"]
    },
    "food_safety": {
        "en": ["What food safety precautions should donors follow?", "Can I donate expired food?", "How should surplus food be handled safely?", "What should I check before donating food?"],
        "hi": ["दाता को कौन सी खाद्य सुरक्षा सावधानियाँ रखनी चाहिए?", "क्या मैं समाप्त हो चुका भोजन दान कर सकता हूँ?", "बचे हुए भोजन को सुरक्षित कैसे रखें?", "भोजन दान करने से पहले क्या जाँचें?"],
        "kn": ["ದಾನಿಗಳು ಯಾವ ಆಹಾರ ಸುರಕ್ಷತಾ ಮುನ್ನೆಚ್ಚರಿಕೆಗಳನ್ನು ಪಾಲಿಸಬೇಕು?", "ಅವಧಿ ಮೀರಿದ ಆಹಾರವನ್ನು ದಾನ ಮಾಡಬಹುದೇ?", "ಉಳಿದ ಆಹಾರವನ್ನು ಸುರಕ್ಷಿತವಾಗಿ ಹೇಗೆ ನಿರ್ವಹಿಸಬೇಕು?", "ಆಹಾರ ದಾನ ಮಾಡುವ ಮೊದಲು ಏನು ಪರಿಶೀಲಿಸಬೇಕು?"]
    },
    "role_permissions": {
        "en": ["What can an NGO do on FoodBridge?", "What are the volunteer driver's responsibilities?", "What can a restaurant donor manage?", "What does the administrator do?"],
        "hi": ["फूडब्रिज पर एनजीओ क्या कर सकता है?", "स्वयंसेवी ड्राइवर की जिम्मेदारियाँ क्या हैं?", "रेस्टोरेंट दाता क्या प्रबंधित कर सकता है?", "एडमिनिस्ट्रेटर क्या करता है?"],
        "kn": ["ಫುಡ್‌ಬ್ರಿಡ್ಜ್‌ನಲ್ಲಿ ಎನ್‌ಜಿಒ ಏನು ಮಾಡಬಹುದು?", "ಸ್ವಯಂಸೇವಕ ಚಾಲಕನ ಜವಾಬ್ದಾರಿಗಳೇನು?", "ರೆಸ್ಟೋರೆಂಟ್ ದಾನಿ ಏನು ನಿರ್ವಹಿಸಬಹುದು?", "ನಿರ್ವಾಹಕರು ಏನು ಮಾಡುತ್ತಾರೆ?"]
    },
    "contact_support": {
        "en": ["How do I contact support?", "I need help from the support team", "Where can I report a problem?", "Can someone from FoodBridge help me?"],
        "hi": ["सपोर्ट से कैसे संपर्क करूँ?", "मुझे सहायता टीम से मदद चाहिए", "समस्या की शिकायत कहाँ करूँ?", "क्या फूडब्रिज से कोई मेरी मदद कर सकता है?"],
        "kn": ["ಸಹಾಯ ತಂಡವನ್ನು ಹೇಗೆ ಸಂಪರ್ಕಿಸುವುದು?", "ನನಗೆ ಸಹಾಯ ತಂಡದ ನೆರವು ಬೇಕು", "ಸಮಸ್ಯೆಯನ್ನು ಎಲ್ಲಿ ವರದಿ ಮಾಡಬಹುದು?", "ಫುಡ್‌ಬ್ರಿಡ್ಜ್‌ನಿಂದ ಯಾರಾದರೂ ಸಹಾಯ ಮಾಡಬಹುದೇ?"]
    },
    "out_of_scope": {
        "en": ["Tell me a joke about aliens", "Who will win the next cricket match?", "Write me a romantic poem", "What is the capital of France?"],
        "hi": ["एलियंस पर एक मज़ाक सुनाओ", "अगला क्रिकेट मैच कौन जीतेगा?", "मेरे लिए एक प्रेम कविता लिखो", "फ्रांस की राजधानी क्या है?"],
        "kn": ["ಏಲಿಯನ್ಸ್ ಬಗ್ಗೆ ಒಂದು ಜೋಕ್ ಹೇಳಿ", "ಮುಂದಿನ ಕ್ರಿಕೆಟ್ ಪಂದ್ಯವನ್ನು ಯಾರು ಗೆಲ್ಲುತ್ತಾರೆ?", "ನನಗಾಗಿ ಪ್ರೇಮ ಕವಿತೆ ಬರೆಯಿರಿ", "ಫ್ರಾನ್ಸ್‌ನ ರಾಜಧಾನಿ ಯಾವುದು?"]
    }
}

# Convert the hand-written FoodBridge examples to a dataframe.
rows = []
for intent, langs in examples.items():
    for language, messages in langs.items():
        for message in messages:
            rows.append({
                "text": message.strip(),
                "intent": intent,
                "language": language,
                "source": "foodbridge_custom"
            })

foodbridge_df = pd.DataFrame(rows).drop_duplicates(subset=["text"]).reset_index(drop=True)

# Load selected examples from MASSIVE. datasets<4 is installed above because
# the published dataset repository uses a dataset loading script.
from datasets import load_dataset

MASSIVE_CONFIGS = {"en-US": "en", "hi-IN": "hi", "kn-IN": "kn"}
MASSIVE_PER_CLASS_PER_LANGUAGE = 20
MASSIVE_OUT_OF_SCOPE_PER_LANGUAGE = 40

massive_rows = []
massive_errors = []

for config, language_code in MASSIVE_CONFIGS.items():
    print(f"Loading MASSIVE {config}...")
    try:
        massive_split = load_dataset(
            "AmazonScience/massive",
            config,
            split="train",
            trust_remote_code=True
        )
        intent_feature = massive_split.features["intent"]
        for item in massive_split:
            utterance = str(item.get("utt", "")).strip()
            if not utterance:
                continue

            raw_intent = item.get("intent")
            if hasattr(intent_feature, "int2str") and isinstance(raw_intent, (int, np.integer)):
                intent_name = intent_feature.int2str(int(raw_intent)).lower()
            else:
                intent_name = str(raw_intent).lower()

            # MASSIVE's labels are not FoodBridge labels. Only map clear
            # conversational matches; other assistant intents are out of scope.
            if "greet" in intent_name:
                mapped_intent = "greeting"
            elif "thank" in intent_name:
                mapped_intent = "thanks"
            elif "bye" in intent_name or "goodbye" in intent_name:
                mapped_intent = "goodbye"
            else:
                mapped_intent = "out_of_scope"

            massive_rows.append({
                "text": utterance,
                "intent": mapped_intent,
                "language": language_code,
                "source": "massive"
            })
    except Exception as exc:
        massive_errors.append((config, repr(exc)))
        print(f"Could not load {config}: {exc}")

massive_df = pd.DataFrame(massive_rows)

if massive_df.empty:
    raise RuntimeError(
        "MASSIVE could not be loaded for any language. Check the Colab internet connection "
        "and the dataset-loading error above, then rerun this cell."
    )

# Keep a controlled number of examples per mapped intent and language.
sampled_parts = []
for (intent, language), group in massive_df.groupby(["intent", "language"]):
    cap = MASSIVE_OUT_OF_SCOPE_PER_LANGUAGE if intent == "out_of_scope" else MASSIVE_PER_CLASS_PER_LANGUAGE
    sampled_parts.append(group.sample(n=min(cap, len(group)), random_state=SEED))

massive_sampled_df = pd.concat(sampled_parts, ignore_index=True)

# Combine both sources, remove duplicate text, shuffle deterministically.
df = pd.concat([foodbridge_df, massive_sampled_df], ignore_index=True)
df = df.drop_duplicates(subset=["text"]).sample(frac=1, random_state=SEED).reset_index(drop=True)

os.makedirs("/content/foodbridge_data", exist_ok=True)
df.to_csv("/content/foodbridge_data/foodbridge_combined_dataset.csv", index=False, encoding="utf-8")
foodbridge_df.to_csv("/content/foodbridge_data/foodbridge_custom_examples.csv", index=False, encoding="utf-8")
massive_sampled_df.to_csv("/content/foodbridge_data/massive_selected_examples.csv", index=False, encoding="utf-8")

print("Custom FoodBridge examples:", len(foodbridge_df))
print("Selected MASSIVE examples:", len(massive_sampled_df))
print("Combined dataset rows:", len(df))
print("Number of intent labels:", df["intent"].nunique())
print("\\nRows by language and source:")
display(df.groupby(["language", "source"]).size().unstack(fill_value=0))
print("\\nExamples per intent:")
display(df.groupby(["intent", "language"]).size().unstack(fill_value=0))
display(df.sample(min(12, len(df)), random_state=SEED))

if massive_errors:
    print("\\nMASSIVE loading issues:", massive_errors)


## 2. Prepare train, validation, and test sets

We split the combined data into training, validation, and held-out test sets using stratification by intent. The held-out set measures intent classification on data the model did not train on. Because the custom FoodBridge examples are still relatively small, inspect the per-intent metrics and add reviewed examples for weak classes before deployment.


In [ ]:
label_names = sorted(df["intent"].unique())
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}
df["label"] = df["intent"].map(label2id)

# Stratified splits preserve intent proportions.
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
print("Label count:", len(label_names))

## 3. Load the multilingual pretrained model

This notebook uses `google-bert/bert-base-multilingual-cased`. It is a multilingual transformer that can be fine-tuned for FoodBridge intent classification. The first download requires internet access.

In [ ]:
MODEL_NAME = "google-bert/bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

print("Loaded:", MODEL_NAME)
print("Device:", DEVICE)

## 4. Tokenize the datasets

In [ ]:
def prepare_dataset(frame):
    ds = Dataset.from_pandas(
        frame[["text", "label"]].reset_index(drop=True),
        preserve_index=False
    )
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)
    return ds.map(tokenize_batch, batched=True, remove_columns=["text"])

train_ds = prepare_dataset(train_df)
val_ds = prepare_dataset(val_df)
test_ds = prepare_dataset(test_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)

## 5. Fine-tune the classifier

This is the actual training step. GPU training is used automatically when a CUDA GPU is available. The model is selected using validation macro F1, not the test set.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

training_args = TrainingArguments(
    output_dir="/content/foodbridge_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

train_result = trainer.train()
print("Training metrics:", train_result.metrics)

## 6. Evaluate on the held-out test set

In [ ]:
test_output = trainer.predict(test_ds)
test_logits = test_output.predictions
test_predictions = np.argmax(test_logits, axis=1)
test_labels = test_output.label_ids

print("Held-out test classification report:")
print(classification_report(
    test_labels,
    test_predictions,
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0
))

cm = confusion_matrix(
    test_labels, test_predictions, labels=list(range(len(label_names)))
)
plt.figure(figsize=(14, 11))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=label_names, yticklabels=label_names, cmap="Blues"
)
plt.xlabel("Predicted intent")
plt.ylabel("True intent")
plt.title("FoodBridge intent classifier — held-out test set")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Evaluate each language separately

This reports performance on the held-out test examples for English, Hindi, and Kannada. Remember that general examples from MASSIVE and custom FoodBridge examples come from different sources; check the per-intent results and the misclassified examples rather than relying on accuracy alone.


In [ ]:
test_results = test_df.reset_index(drop=True).copy()
test_results["true_label"] = test_labels
test_results["predicted_label"] = test_predictions
test_results["predicted_intent"] = test_results["predicted_label"].map(id2label)

language_rows = []
for lang in ["en", "hi", "kn"]:
    subset = test_results[test_results["language"] == lang]
    if subset.empty:
        continue
    language_rows.append({
        "language": lang,
        "n_test": len(subset),
        "accuracy": accuracy_score(subset["true_label"], subset["predicted_label"]),
        "macro_f1": f1_score(
            subset["true_label"], subset["predicted_label"],
            labels=list(range(len(label_names))),
            average="macro", zero_division=0
        )
    })

language_metrics = pd.DataFrame(language_rows)
display(language_metrics)

print("Misclassified examples:")
display(test_results[
    test_results["true_label"] != test_results["predicted_label"]
][["text", "language", "intent", "predicted_intent"]])

## 8. Save the model, tokenizer, labels, and evaluation results

The output folder contains the model artifacts required for inference. Do not commit large weights to Git unless you intend to store them there.

In [ ]:
MODEL_DIR = "/content/foodbridge_model"
os.makedirs(MODEL_DIR, exist_ok=True)

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

with open(os.path.join(MODEL_DIR, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()}
        },
        f, ensure_ascii=False, indent=2
    )

test_metrics = {
    "accuracy": float(accuracy_score(test_labels, test_predictions)),
    "precision_macro": float(precision_recall_fscore_support(
        test_labels, test_predictions, average="macro", zero_division=0
    )[0]),
    "recall_macro": float(precision_recall_fscore_support(
        test_labels, test_predictions, average="macro", zero_division=0
    )[1]),
    "f1_macro": float(f1_score(
        test_labels, test_predictions, labels=list(range(len(label_names))),
        average="macro", zero_division=0
    ))
}

with open(os.path.join(MODEL_DIR, "test_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

language_metrics.to_csv(os.path.join(MODEL_DIR, "language_metrics.csv"), index=False)
df.drop(columns=["label"], errors="ignore").to_csv(
    os.path.join(MODEL_DIR, "combined_training_dataset.csv"), index=False, encoding="utf-8"
)

print("Saved artifacts:", os.listdir(MODEL_DIR))
print("Held-out test metrics:", test_metrics)

dataset_metadata = {
    "primary_dataset": "Custom FoodBridge multilingual examples",
    "supplementary_dataset": "AmazonScience/MASSIVE",
    "massive_languages": ["en-US", "hi-IN", "kn-IN"],
    "model": MODEL_NAME,
    "task": "multilingual intent classification",
    "note": "MASSIVE general intents are mapped to greeting/thanks/goodbye; other MASSIVE intents are mapped to out_of_scope. FoodBridge-specific intents come from custom examples."
}
with open(os.path.join(MODEL_DIR, "dataset_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, ensure_ascii=False, indent=2)


## 9. Try sample predictions

In [ ]:
import torch.nn.functional as F

model = trainer.model
model.eval()

def predict_intent(text):
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=128
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]
    idx = int(torch.argmax(probs).item())
    return {
        "message": text,
        "intent": id2label[idx],
        "confidence": round(float(probs[idx].item()), 4)
    }

samples = [
    "How can I post surplus food?",
    "मैं अपने एनजीओ की मंजूरी कैसे जाँचूँ?",
    "ನನ್ನ ಪಿಕಪ್ ನಿಯೋಜನೆಯನ್ನು ಹೇಗೆ ನೋಡುವುದು?",
    "How do I check delivery status?",
    "Tell me a joke about aliens"
]
for sample in samples:
    print(predict_intent(sample))

## 10. Export the model ZIP

Download this ZIP and extract it into your FoodBridge project, for example under `chatbot/model/`. The application must load these files for inference; it must not depend on the Colab session remaining active.

**Note:** the model is a classifier. In FoodBridge, map each predicted intent to a verified response in the detected language. Use authenticated existing backend endpoints for live donation and delivery details.

In [ ]:
ZIP_PATH = shutil.make_archive(
    "/content/foodbridge_model_export",
    "zip",
    root_dir="/content",
    base_dir="foodbridge_model"
)
print("Created:", ZIP_PATH)

from google.colab import files
files.download(ZIP_PATH)

## Integration checklist

- Extract the exported model ZIP into your FoodBridge project.
- Load the saved tokenizer and model in a Python inference service, or keep model inference as a separate service that your existing Express backend calls.
- Use the predicted intent to select a reviewed response in the user's language.
- For private, live information (such as a user's actual donation or delivery status), query the authenticated FoodBridge backend; do not invent status from the intent classifier.
- Add a confidence threshold and a safe fallback when the model is uncertain.
- This is a **text chatbot model**, not a voice chatbot. Voice can be added later using speech-to-text before intent prediction and text-to-speech for the reply.

### Dataset citation

Amazon Science. *MASSIVE: A 1M-Example Multilingual Natural Language Understanding Dataset with 52 Typologically-Diverse Languages.* Dataset card and data: https://huggingface.co/datasets/AmazonScience/massive. License: CC BY 4.0. See the dataset card for the associated research paper and citation details.
